# Beyond the Single Text: NLP Reading in Digital Humanities

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 3 · Wednesday 17 June 2026

**Lecturer:** Isuri Anuradha

> Before running anything, make sure the kernel is **NLPAICS 09** (menu: *Kernel → Change Kernel*). It should already be selected.

## 0 · Environment check

In [ ]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 09"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")


In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import spacy
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import HTML, display
from huggingface_hub import login
from transformers import pipeline, BitsAndBytesConfig
import torch

print("All imports successful ✓")

## LLM Setup — HuggingFace Transformers
How to create a huggingface access token ([https://huggingface.co/settings/](https://))
- You have to provide access for public generated repository access
- Switch `MODEL_ID` to `google/gemma-2b-it` if preferred. ( `meta-llama/Meta-Llama-3-8B-Instruct`)

In [ ]:
login(token="<<ACCESS TOKEN>>")
print("HuggingFace authenticated ✓")

In [ ]:
# ── LLM-A. Load model ─────────────────────────────────────────────────────────
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
MODEL_ID="google/gemma-2b-it"

# # 4-bit quantisation
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )
pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.float16,
    # model_kwargs={"quantization_config": bnb_config},
    device_map="auto",
    max_new_tokens=512,
)
print(f"Model loaded: {MODEL_ID} ✓")

In [ ]:
# ── LLM-B. Core call function ─────────────────────────────────────────────────
# def call_llm(prompt, max_new_tokens=512):
#     """Send a prompt to the loaded HuggingFace model and return the response text."""
#     formatted = f"[INST] {prompt} [/INST]"
#     result = pipe(
#         formatted,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         temperature=0.1,
#         return_full_text=False,   # return only the generated part
#     )
#     return result[0]["generated_text"].strip()

# # Quick test
# print(call_llm("Reply with only: LLM ready ✓"))

# ── LLM-B. Core call function (Gemma format) ──────────────────────────────────
def call_llm(prompt, max_new_tokens=512):
    """Format prompt using Gemma's chat template and return the response."""
    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    result = pipe(
        formatted,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )
    # Strip Gemma's end-of-turn token if present
    text = result[0]["generated_text"].strip()
    return text.replace("<end_of_turn>", "").strip()

# Quick test
print(call_llm("Reply with exactly three words: LLM is ready"))

1. Sample of History Text

In [ ]:
# ── Replace this with your own corpus ────────────────────────────────────────
HISTORICAL_TEXT = """
The Battle of Waterloo was fought on 18 June 1815 near Waterloo in present-day Belgium.
Napoleon Bonaparte, Emperor of France, suffered a decisive defeat at the hands of the
Duke of Wellington commanding the Anglo-allied army and Field Marshal Blücher leading
the Prussian Army. The battle ended the Napoleonic Wars and resulted in Napoleon's exile
to Saint Helena.

Following Waterloo, the Congress of Vienna, convened in Vienna in 1814, reshaped the
political boundaries of Europe. Klemens von Metternich of Austria played a central role
in the congress negotiations. Prussia gained territories along the Rhine, while Britain
consolidated its colonial holdings across India and Africa.

In 1857, the Indian Rebellion broke out in Meerut and quickly spread to Delhi, Lucknow,
and Kanpur. Rani Lakshmibai of Jhansi became a symbol of resistance against the British
East India Company. The rebellion led to the Government of India Act 1858, dissolving
the company and transferring power to the British Crown.

Cecil Rhodes founded the British South Africa Company in 1889 and was instrumental in
expanding British influence into Rhodesia. Rhodes served as Prime Minister of the Cape
Colony from 1890 to 1896. The discovery of diamonds in Kimberley and gold in
Johannesburg fuelled rapid European migration to southern Africa.

Florence Nightingale transformed nursing practices during the Crimean War fought between
1853 and 1856. She established a hospital at Scutari, near Constantinople, and
dramatically reduced mortality rates among British soldiers. In 1860 she founded the
Nightingale Training School at St Thomas' Hospital in London.
"""

print(f"Text length: {len(HISTORICAL_TEXT.split())} words")

2 · NER — spaCy + LLM Post-correction

Where LLM is used: After spaCy extracts entities, the LLM reviews and corrects misclassifications — especially important for historical names, titles, and place spellings not in spaCy's training data.

In [ ]:
# ── 2a. spaCy NER ─────────────────────────────────────────────────────────────
nlp = spacy.load("en_core_web_lg")
doc = nlp(HISTORICAL_TEXT)

ENTITY_TYPES = {"PERSON", "ORG", "GPE", "LOC", "DATE", "EVENT", "NORP", "FAC"}

entities = []
for ent in doc.ents:
    if ent.label_ in ENTITY_TYPES:
        entities.append({
            "text":  ent.text.strip(),
            "label": ent.label_,
            "start": ent.start_char,
            "end":   ent.end_char,
        })

ent_df = pd.DataFrame(entities).drop_duplicates(subset=["text", "label"])
print(f"spaCy found {len(ent_df)} unique entities")
ent_df.sort_values("label")

In [ ]:
# ── 2b. LLM post-correction ───────────────────────────────────────────────────
def verify_entities_llm(text, spacy_entities):
    # Format entity list clearly for Gemma
    ent_formatted = "\n".join(
        f'  {i+1}. "{e["text"]}" → {e["label"]}'
        for i, e in enumerate(spacy_entities)
    )

    prompt = f"""You are a historical NER expert specialising in toponyms and historical place names.

Below is a numbered list of ALL entities extracted from a historical text.
Your job is to review EVERY entity and fix wrong labels where needed.

Pay special attention to:
- Toponyms (place names, regions, countries, cities) → must be GPE or LOC
- Historical place names that spaCy may have mislabelled as ORG or NORP
- Person titles that got split from names
- Organisations vs geopolitical entities (e.g. "British East India Company" is ORG, "Britain" is GPE)

Text: {text[:500]}

Entity list to review (ALL {len(spacy_entities)} must appear in output):
{ent_formatted}

Return the COMPLETE corrected list as a JSON array.
Every entity from the input must appear in the output — do not drop any.
Only change the label if it is wrong. Keep correct labels unchanged.
No explanation. No markdown. Output JSON only:
[{{"text": "...", "label": "..."}}]"""

    response = call_llm(prompt, max_new_tokens=800)
    print(f"Raw response:\n{response}\n")

    # ── Fix common Gemma JSON errors ──────────────────────────────────────────
    # Remove markdown fences
    cleaned = re.sub(r"```(?:json)?|```", "", response).strip()

    # Fix duplicate closing braces e.g. }}] → }]
    cleaned = re.sub(r'\}\s*\}(\s*[,\]])', r'}\1', cleaned)

    # Extract the JSON array
    match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if match:
        json_str = match.group()
        # Fix trailing comma before ] e.g. {...},] → {...}]
        json_str = re.sub(r',\s*\]', ']', json_str)
        try:
            result = json.loads(json_str)
            if len(result) < len(spacy_entities) * 0.5:
                print(f"⚠ LLM returned only {len(result)}/{len(spacy_entities)} entities — merging with spaCy output")
                # Merge: LLM corrections override spaCy for matching entities
                spacy_map = {e["text"]: e["label"] for e in spacy_entities}
                llm_map   = {e["text"]: e["label"] for e in result}
                spacy_map.update(llm_map)  # LLM corrections win
                return [{"text": t, "label": l} for t, l in spacy_map.items()]
            return result
        except json.JSONDecodeError as err:
            print(f"⚠ JSON parse error: {err}")

    # Fallback — extract key:value pairs directly from raw text
    fallback = re.findall(
        r'"text"\s*:\s*"([^"]+)"\s*,\s*"label"\s*:\s*"([^"]+)"', cleaned
    )
    if fallback:
        print(f"⚠ Used fallback parser — got {len(fallback)} entities")
        # Merge fallback with original spaCy entities
        spacy_map = {e["text"]: e["label"] for e in spacy_entities}
        for text_val, label in fallback:
            spacy_map[text_val] = label
        return [{"text": t, "label": l} for t, l in spacy_map.items()]

    print("⚠ All parse attempts failed — keeping spaCy entities unchanged")
    return spacy_entities


print("Running LLM entity verification...")
corrected_ents = verify_entities_llm(HISTORICAL_TEXT, entities)
corrected_df = (
    pd.DataFrame(corrected_ents)
    .drop_duplicates(subset=["text", "label"])
    .sort_values("label")
    .reset_index(drop=True)
)

# ── Show what changed ──────────────────────────────────────────────────────────
spacy_map     = {e["text"]: e["label"] for e in entities}
corrected_map = {e["text"]: e["label"] for e in corrected_ents}

changes = [
    {"entity": text, "spaCy label": spacy_map[text], "LLM label": corrected_map[text]}
    for text in spacy_map
    if text in corrected_map and spacy_map[text] != corrected_map[text]
]

print(f"\nspaCy: {len(entities)} → LLM corrected: {len(corrected_df)} entities")
if changes:
    print(f"\nLabel corrections made ({len(changes)}):")
    print(pd.DataFrame(changes).to_string(index=False))
else:
    print("No label changes made")

corrected_df

In [ ]:
# ── 2c. Build entity type lookup (used throughout notebook) ───────────────────
ent_type_map = {}
for ent in doc.ents:
    if ent.label_ in ENTITY_TYPES:
        ent_type_map[ent.text.strip()] = ent.label_

# Merge LLM corrections into lookup
for ent in corrected_ents:
    ent_type_map[ent["text"]] = ent["label"]

# Use corrected entities going forward
final_ents = corrected_df if len(corrected_df) > 0 else ent_df

print(f"Entity type map built: {len(ent_type_map)} entries ✓")

In [ ]:
# ── 2d. Entity frequency chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = final_ents["label"].value_counts()
colors = plt.cm.Set2.colors
type_counts.plot(kind="bar", ax=axes[0], color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_title("Entity Count by Type (LLM corrected)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Entity Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=30)

all_ent_texts = [ent.text.strip() for ent in doc.ents if ent.label_ in ENTITY_TYPES]
top_ents = Counter(all_ent_texts).most_common(12)
labels, vals = zip(*top_ents)
axes[1].barh(labels[::-1], vals[::-1], color="steelblue", edgecolor="white")
axes[1].set_title("Top 12 Entity Mentions", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Frequency")

plt.tight_layout()
plt.show()
print("NER chart displayed ✓")

## 3 · Relation Extraction — Pattern + LLM

---
**Where LLM is used:** Rule-based RE misses nuanced historical phrasing. The LLM reads each sentence and extracts structured subject–relation–object triples, which are merged with pattern-based results.

In [ ]:
# ── 3a. Pattern-based RE (baseline) ──────────────────────────────────────────
RELATION_PATTERNS = [
    (r"([A-Z][\w\s]+)\s+(?:fought|battled|defeated)\s+(?:at|near|in)\s+([A-Z][\w\s]+)",  "FOUGHT_AT"),
    (r"([A-Z][\w\s]+)\s+(?:founded|established|created)\s+(?:the\s+)?([A-Z][\w\s]+)",      "FOUNDED"),
    (r"([A-Z][\w\s]+)\s+(?:served as|was|became)\s+([A-Z][\w\s]+(?:of|in)\s+[A-Z][\w\s]+)","HELD_ROLE"),
    (r"([A-Z][\w\s]+)\s+(?:led|commanded)\s+(?:the\s+)?([A-Z][\w\s]+)",                    "COMMANDED"),
    (r"([A-Z][\w\s]+)\s+(?:gained|acquired|annexed)\s+([A-Z][\w\s]+)",                     "ACQUIRED"),
]

def extract_pattern_relations(text):
    results = []
    for pattern, label in RELATION_PATTERNS:
        for m in re.finditer(pattern, text):
            groups = [g.strip() for g in m.groups() if g]
            if len(groups) >= 2:
                results.append({
                    "subject":  groups[0],
                    "relation": label,
                    "object":   groups[1],
                    "source":   "pattern",
                })
    return results

pattern_rels = extract_pattern_relations(HISTORICAL_TEXT)
print(f"Pattern-based relations: {len(pattern_rels)}")

In [ ]:
# ── 3b. LLM relation extraction (per sentence) ───────────────────────────────
def extract_relations_llm(sentence):
    """
    Ask the LLM to extract structured triples from a single sentence.
    Returns a list of {subject, relation, object} dicts.
    """
    prompt = f"""Extract all entity relationships from this historical sentence.
Return ONLY a valid JSON list — no explanation, no markdown backticks:
[
  {{"subject": "Napoleon Bonaparte", "relation": "DEFEATED_AT", "object": "Waterloo"}}
]

Use relation labels like: DEFEATED_AT, FOUNDED, COMMANDED, EXILED_TO, HELD_ROLE,
LOCATED_IN, SPREAD_TO, RESISTED, ESTABLISHED, ASSOCIATED_WITH

If no clear relation exists, return: []

Sentence: {sentence}"""

    response = call_llm(prompt, max_new_tokens=300)
    match = re.search(r'\[.*?\]', response, re.DOTALL)
    if match:
        try:
            rels = json.loads(match.group())
            for r in rels:
                r["source"] = "llm"
            return rels
        except json.JSONDecodeError:
            pass
    return []


print("Running LLM relation extraction (one call per sentence)...")
llm_rels = []
sentences = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 20]

for i, sent in enumerate(sentences):
    rels = extract_relations_llm(sent)
    llm_rels.extend(rels)
    print(f"  Sentence {i+1}/{len(sentences)}: {len(rels)} relations found")

print(f"\nLLM extracted: {len(llm_rels)} relations")

In [ ]:
# ── 3c. Merge pattern + LLM relations ────────────────────────────────────────
all_rels = pattern_rels + llm_rels
rel_df = pd.DataFrame(all_rels).drop_duplicates(subset=["subject", "relation", "object"])

print(f"Total unique relations: {len(rel_df)}")
print(f"  Pattern-based: {len(rel_df[rel_df.source=='pattern'])}")
print(f"  LLM-extracted: {len(rel_df[rel_df.source=='llm'])}")
rel_df.head(15)

In [ ]:
# ── 4a. Build NetworkX graph ──────────────────────────────────────────────────
TYPE_COLORS = {
    "PERSON": "#E05C5C", "ORG": "#5C8FE0", "GPE": "#5CB85C",
    "LOC": "#3DA65A",    "DATE": "#F0AD4E", "EVENT": "#9B59B6",
    "NORP": "#1ABC9C",   "FAC": "#E67E22",  "OTHER": "#95A5A6",
}

G = nx.DiGraph()

def get_type(name):
    return ent_type_map.get(name, "OTHER")

def add_relation(g, subj, rel, obj):
    s_type = get_type(subj)
    o_type = get_type(obj)
    g.add_node(subj, entity_type=s_type, color=TYPE_COLORS.get(s_type, "#95A5A6"))
    g.add_node(obj,  entity_type=o_type, color=TYPE_COLORS.get(o_type, "#95A5A6"))
    if g.has_edge(subj, obj):
        g[subj][obj]["relations"].add(rel)
        g[subj][obj]["weight"] += 1
    else:
        g.add_edge(subj, obj, relations={rel}, weight=1)

for _, row in rel_df.iterrows():
    add_relation(G, row["subject"], row["relation"], row["object"])

print(f"Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges ✓")

In [ ]:
# ── 4b. Matplotlib static KG ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 12))
# fig.patch.set_facecolor("#1A1A2E")
# ax.set_facecolor("#1A1A2E")

pos = nx.spring_layout(G, k=2.5, seed=42)
node_colors = [d.get("color", "#95A5A6") for _, d in G.nodes(data=True)]
node_sizes  = [300 + 200 * G.degree(n) for n in G.nodes()]
edge_weights= [d.get("weight", 1) for _, _, d in G.edges(data=True)]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.92, ax=ax)
nx.draw_networkx_edges(G, pos, width=[w*0.9 for w in edge_weights],
                       edge_color="#AAAAAA", alpha=0.5, arrows=True,
                       arrowstyle="->", arrowsize=15, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, font_color="black", font_weight="bold", ax=ax)

edge_labels = {(u,v): ", ".join(d["relations"]) for u,v,d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=5.5, font_color="#F0AD4E", ax=ax)

legend_patches = [mpatches.Patch(color=c, label=t) for t,c in TYPE_COLORS.items()
                  if t in {d.get("entity_type") for _,d in G.nodes(data=True)}]
ax.legend(handles=legend_patches, loc="upper left", fontsize=8,
          facecolor="#1A1A2E", edgecolor="grey", labelcolor="white")

ax.set_title("Historical Knowledge Graph (LLM + Pattern RE)",
             color="white", fontsize=16, fontweight="bold", pad=15)
ax.axis("off")
plt.tight_layout()
plt.show()
print("Knowledge graph displayed ✓")

In [ ]:
# ── 4c. Graph analytics — PageRank importance ─────────────────────────────────
pagerank = nx.pagerank(G, alpha=0.85)
degree_c = nx.degree_centrality(G.to_undirected())

analytics_df = pd.DataFrame({
    "Node":        list(pagerank.keys()),
    "PageRank":    list(pagerank.values()),
    "Degree":      [degree_c[n] for n in pagerank],
    "Entity Type": [G.nodes[n].get("entity_type", "OTHER") for n in pagerank],
}).sort_values("PageRank", ascending=False)

print("Top 10 entities by historical importance (PageRank):")
analytics_df.head(10).round(4)

## 5 · LLM Knowledge Graph Narrative

**Where LLM is used:** Generates a coherent historical narrative from the KG edges — useful for automated report generation.

In [ ]:
# ── 5. LLM Coreference Resolution (generic, no registry) ─────────────────────

def resolve_coreferences_llm(text):
    """
    Two-pass approach:
    Pass 1 — Ask LLM to build a coreference map from the full text
    Pass 2 — Apply the map sentence by sentence
    """

    # ── Pass 1: Build coreference map ────────────────────────────────────────
    map_prompt = f"""Read this historical text and identify every pronoun and vague reference.
For each one, identify what entity it refers to.

Text:
{text}

Output ONLY a JSON object mapping each reference to its entity.
No explanation. No markdown.
Example format:
{{
  "She": "Florence Nightingale",
  "she": "Florence Nightingale",
  "he": "Cecil Rhodes",
  "the company": "British East India Company",
  "the rebellion": "Indian Rebellion",
  "the battle": "Battle of Waterloo",
  "the congress": "Congress of Vienna",
  "the school": "Nightingale Training School",
  "the hospital": "Scutari hospital",
  "the war": "Crimean War"
}}"""

    print("Pass 1: Building coreference map...")
    map_response = call_llm(map_prompt, max_new_tokens=400)
    print(f"Map response:\n{map_response}\n")

    # Parse the coreference map
    coref_map = {}
    try:
        # Try direct JSON parse
        cleaned = re.sub(r"```(?:json)?|```", "", map_response).strip()
        match   = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if match:
            coref_map = json.loads(match.group())
            print(f"✓ Coreference map built: {len(coref_map)} entries")
            for k, v in coref_map.items():
                print(f"  '{k}' → '{v}'")
    except json.JSONDecodeError:
        # Fallback: parse key:value pairs directly
        pairs = re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', map_response)
        coref_map = dict(pairs)
        print(f"✓ Fallback map built: {len(coref_map)} entries")

    if not coref_map:
        print("⚠ Could not build coreference map — returning original text")
        return text

    # ── Pass 2: Resolve each sentence using the map ───────────────────────────
    def resolve_sentence(sentence, coref_map):
        """Apply coreference map to a single sentence via LLM."""

        # Only call LLM if sentence contains a known reference
        lower = sentence.lower()
        needs_resolution = any(
            ref.lower() in lower
            for ref in coref_map.keys()
        )
        if not needs_resolution:
            return sentence

        map_str = "\n".join(f'  "{k}" → "{v}"' for k, v in coref_map.items())

        prompt = f"""Replace the references in this sentence using the mapping below.
Only replace references that appear in the mapping.
Keep all other wording exactly the same.
Output ONLY the rewritten sentence, nothing else.

Coreference mapping:
{map_str}

Sentence:
{sentence}

Rewritten sentence:"""

        result = call_llm(prompt, max_new_tokens=150)

        # Clean up Gemma output
        result = result.strip().strip('"').strip("'")

        # Sanity check — if output is wildly different length, discard
        if len(result) < len(sentence) * 0.5 or len(result) > len(sentence) * 2.5:
            print(f"  ⚠ Discarded (length mismatch): {result[:60]}...")
            return sentence

        return result

    # ── Apply to all sentences ────────────────────────────────────────────────
    print("\nPass 2: Resolving sentences...")
    sentences = [s.strip() for s in text.strip().split(".") if s.strip()]
    resolved  = []
    changes   = 0

    for i, sent in enumerate(sentences):
        sent_full     = sent + "."
        resolved_sent = resolve_sentence(sent_full, coref_map)

        if resolved_sent != sent_full:
            changes += 1
            print(f"\n  [{i+1}] ORIGINAL : {sent_full}")
            print(f"       RESOLVED : {resolved_sent}")
        else:
            resolved.append(resolved_sent)
            continue

        resolved.append(resolved_sent)

    print(f"\n── {changes}/{len(sentences)} sentences updated ──")
    return " ".join(resolved)


# ── Run ───────────────────────────────────────────────────────────────────────
print("Running generic LLM coreference resolution...\n")
RESOLVED_TEXT = resolve_coreferences_llm(HISTORICAL_TEXT)

print("\n── Final Resolved Text ───────────────────────────────────────────────")
print(RESOLVED_TEXT)

In [ ]:
# ── 5b. Re-run NER and RE on resolved text ────────────────────────────────────
print("Re-running spaCy on coreference-resolved text...")
doc_resolved = nlp(RESOLVED_TEXT)

# Extract fresh entities from resolved text
entities_resolved = []
for ent in doc_resolved.ents:
    if ent.label_ in ENTITY_TYPES:
        entities_resolved.append({
            "text":  ent.text.strip(),
            "label": ent.label_,
        })

ent_df_resolved = (
    pd.DataFrame(entities_resolved)
    .drop_duplicates(subset=["text", "label"])
    .sort_values("label")
    .reset_index(drop=True)
)

# Compare entity counts
print(f"\nOriginal text  → {len(ent_df)} unique entities")
print(f"Resolved text  → {len(ent_df_resolved)} unique entities")
print(f"Net gain       → +{len(ent_df_resolved) - len(ent_df)} entities from coreference resolution")

# New entities found only after resolution
original_texts  = set(ent_df["text"].tolist())
resolved_texts  = set(ent_df_resolved["text"].tolist())
new_entities    = resolved_texts - original_texts

if new_entities:
    print(f"\nNew entities surfaced by resolution ({len(new_entities)}):")
    for e in sorted(new_entities):
        label = ent_type_map.get(e, "?")
        print(f"  {e:30s} [{label}]")

ent_df_resolved

In [ ]:
# ── 5b-fix. Post-process resolved text to catch remaining pronouns ────────────

import re

# All pronouns that should never appear as a subject/object in a triple
PRONOUNS = {
    "he", "she", "they", "it", "we", "i", "you",
    "him", "her", "them", "us", "me",
    "his", "hers", "their", "its", "our", "my", "your",
    "himself", "herself", "themselves", "itself",
    "this", "that", "these", "those",
    "who", "whom", "which",
}

def clean_triples(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove or flag triples where subject or object is an unresolved pronoun.
    Also cleans truncated objects, whitespace, and empty strings.
    """
    df = df.copy()

    # Normalise whitespace
    for col in ["subject", "relation", "object"]:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    # Drop rows where subject or object is a bare pronoun
    mask_pronoun_subj = df["subject"].str.lower().isin(PRONOUNS)
    mask_pronoun_obj  = df["object"].str.lower().isin(PRONOUNS)

    n_before = len(df)
    df = df[~mask_pronoun_subj & ~mask_pronoun_obj]
    n_after = len(df)

    print(f"  Removed {n_before - n_after} triples with unresolved pronouns")

    # Drop rows with empty subject, relation, or object
    df = df[(df["subject"] != "") & (df["relation"] != "") & (df["object"] != "")]

    # Drop triples where subject == object (self-loops add no information)
    df = df[df["subject"].str.lower() != df["object"].str.lower()]

    # Drop duplicates again after cleaning
    df = df.drop_duplicates(subset=["subject", "relation", "object"]).reset_index(drop=True)

    return df

In [ ]:
# ── 5c. Rebuild KG from resolved text (relaxed filtering) ────────────────────
print("Re-extracting relations from resolved text...")

pattern_rels_resolved = extract_pattern_relations(RESOLVED_TEXT)

llm_rels_resolved = []
sentences_resolved = [
    s.text.strip() for s in doc_resolved.sents if len(s.text.strip()) > 20
]
for i, sent in enumerate(sentences_resolved):
    rels = extract_relations_llm(sent)
    llm_rels_resolved.extend(rels)
    print(f"  Sentence {i+1}/{len(sentences_resolved)}: {len(rels)} relations")

all_rels_resolved = pattern_rels_resolved + llm_rels_resolved

rel_df_resolved = (
    pd.DataFrame(all_rels_resolved)
    .drop_duplicates(subset=["subject", "relation", "object"])
    .reset_index(drop=True)
)
print(f"\nBefore cleaning : {len(rel_df_resolved)} triples")

# ── Only remove bare pronouns — nothing else ─────────────────────────────────
PRONOUNS = {
    "he","she","they","it","we","i","you",
    "him","her","them","us","me",
    "his","hers","their","its","our","my","your",
    "himself","herself","themselves","itself",
}

mask_bad_subj = rel_df_resolved["subject"].str.lower().str.strip().isin(PRONOUNS)
mask_bad_obj  = rel_df_resolved["object"].str.lower().str.strip().isin(PRONOUNS)
rel_df_resolved = rel_df_resolved[~mask_bad_subj & ~mask_bad_obj].reset_index(drop=True)

print(f"After pronoun filter: {len(rel_df_resolved)} triples")
print()
print(rel_df_resolved[["subject","relation","object"]].to_string())

# ── Build KG ─────────────────────────────────────────────────────────────────
print(f"\nOriginal KG  → {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

G_resolved = nx.DiGraph()
for _, row in rel_df_resolved.iterrows():
    add_relation(G_resolved, row["subject"], row["relation"], row["object"])

print(f"Resolved KG  → {G_resolved.number_of_nodes()} nodes, {G_resolved.number_of_edges()} edges")
print(f"Net gain     → +{G_resolved.number_of_nodes() - G.number_of_nodes()} nodes, "
      f"+{G_resolved.number_of_edges() - G.number_of_edges()} edges")

G   = G_resolved
doc = doc_resolved
print("\n✓ KG updated")

In [ ]:
# ── Knowledge Graph Visualisation (clean, white background) ──────────────────
import networkx as nx
import matplotlib.pyplot as plt

def draw_kg_simple(rel_df, subject_col="subject", predicate_col="relation", object_col="object",
                   figsize=(18, 12), title="Knowledge Graph", save_path="kg.png"):

    df = rel_df[[subject_col, predicate_col, object_col]].copy()
    df.columns = ["subject", "predicate", "object"]
    df["subject"] = df["subject"].str[:35]
    df["object"]  = df["object"].str[:35]

    # ── Build graph ──────────────────────────────────────────────────────────
    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_edge(row["subject"], row["object"], label=row["predicate"])

    # ── Node roles ───────────────────────────────────────────────────────────
    subjects = set(df["subject"])
    objects  = set(df["object"])
    hubs     = subjects & objects

    node_colors = []
    node_sizes  = []
    for n in G.nodes():
        if n in hubs:
            node_colors.append("#a855f7")   # purple  — appears as both
            node_sizes.append(2500)
        elif n in subjects:
            node_colors.append("#3b82f6")   # blue    — source only
            node_sizes.append(2000)
        else:
            node_colors.append("#22c55e")   # green   — target only
            node_sizes.append(1800)

    # ── Layout ───────────────────────────────────────────────────────────────
    pos = nx.spring_layout(G, seed=42, k=2.8, iterations=60)

    # ── Draw ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                           alpha=0.9, edgecolors="#00000033", linewidths=1.2, ax=ax)

    nx.draw_networkx_labels(G, pos, font_size=8, font_color="black",
                            font_weight="bold", ax=ax)

    nx.draw_networkx_edges(G, pos, edge_color="#94a3b8", alpha=0.7,
                           arrows=True, arrowstyle="-|>", arrowsize=20,
                           width=1.5, connectionstyle="arc3,rad=0.15",
                           min_source_margin=25, min_target_margin=25, ax=ax)

    edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                 font_size=7, font_color="#dc2626",
                                 font_weight="bold",
                                 bbox=dict(boxstyle="round,pad=0.25",
                                           fc="white", ec="#e2e8f0", alpha=0.9),
                                 ax=ax)

    # ── Legend ───────────────────────────────────────────────────────────────
    from matplotlib.patches import Patch
    legend = [
        Patch(color="#3b82f6", label="Subject"),
        Patch(color="#22c55e", label="Object"),
        Patch(color="#a855f7", label="Subject + Object (hub)"),
    ]
    ax.legend(handles=legend, loc="upper left", fontsize=9,
              framealpha=0.9, edgecolor="#e2e8f0")

    ax.set_title(title, fontsize=14, fontweight="bold", color="#1e293b", pad=12)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"✓ Saved → {save_path}")


# ── Run ───────────────────────────────────────────────────────────────────────
draw_kg_simple(
    rel_df_resolved,
    subject_col="subject",
    predicate_col="relation",
    object_col="object",
    title="Resolved Knowledge Graph",
    save_path="kg.png",
)

## 6 · Location Extraction & LLM Summaries

**Where LLM is used:** Instead of raw sentence snippets in map popups, the LLM generates a concise historical significance summary for each location.

In [ ]:
# ── 6a. Hardcoded coordinates (Colab-safe — bypasses Nominatim) ───────────────
KNOWN_COORDS = {
    "Waterloo":        (50.711,   4.397),
    "Belgium":         (50.503,   4.469),
    "France":          (46.227,   2.213),
    "Saint Helena":    (-15.965, -5.708),
    "Vienna":          (48.208,  16.373),
    "Austria":         (47.516,  14.550),
    "Prussia":         (52.520,  13.405),
    "India":           (20.593,  78.962),
    "Africa":          (-8.783,  34.508),
    "Meerut":          (28.984,  77.706),
    "Delhi":           (28.613,  77.209),
    "Lucknow":         (26.846,  80.946),
    "Kanpur":          (26.449,  80.331),
    "Jhansi":          (25.448,  78.568),
    "Kimberley":       (-28.728, 24.749),
    "Johannesburg":    (-26.204, 28.047),
    "Rhodesia":        (-19.015, 29.154),
    "Scutari":         (41.008,  29.003),
    "Constantinople":  (41.013,  28.955),
    "London":          (51.507,  -0.127),
    "Cape Colony":     (-33.924, 18.424),
}

location_entities = list({
    ent.text.strip()
    for ent in doc_resolved.ents
    if ent.label_ in {"GPE", "LOC", "FAC"}
})

geocoded = []
for place in location_entities:
    if place in KNOWN_COORDS:
        lat, lon = KNOWN_COORDS[place]
        geocoded.append({"name": place, "lat": lat, "lon": lon})
    else:
        print(f"⚠ No coords for: {place}")

geo_df = pd.DataFrame(geocoded)
print(f"✓ {len(geo_df)} locations ready")
geo_df

In [ ]:
# ── Extract EVENT entities directly from spaCy ────────────────────────────────

# From original spaCy doc
spacy_events = list({
    ent.text.strip()
    for ent in doc_resolved.ents
    if ent.label_ == "EVENT"
})

# Also check ENTITY_TYPES extracted earlier in ent_df
df_events = list(
    ent_df[ent_df["label"] == "EVENT"]["text"].unique()
) if "ent_df" in dir() else []

# Also check resolved doc
resolved_events = list({
    ent.text.strip()
    for ent in doc_resolved.ents
    if ent.label_ == "EVENT"
})

# Also manually add well-known events spaCy often misses
KNOWN_EVENTS = [
    "Battle of Waterloo",
    "Congress of Vienna",
    "Indian Rebellion",
    "Crimean War",
    "Napoleonic Wars",
]

# Merge all sources
event_nodes = list(set(
    spacy_events +
    df_events +
    resolved_events +
    KNOWN_EVENTS
))

print(f"Events from spaCy (original doc)  : {spacy_events}")
print(f"Events from ent_df                : {df_events}")
print(f"Events from resolved doc          : {resolved_events}")
print(f"Events merged (total)             : {len(event_nodes)}")
print(f"Final event list: {event_nodes}")


# ── Add EVENT nodes to KG if missing ─────────────────────────────────────────
TYPE_COLORS = {
    "PERSON": "#E05C5C", "ORG": "#5C8FE0", "GPE":  "#5CB85C",
    "LOC":    "#3DA65A", "DATE": "#F0AD4E", "EVENT": "#9B59B6",
    "NORP":   "#1ABC9C", "FAC": "#E67E22",  "OTHER": "#95A5A6",
}

added = 0
for evt in event_nodes:
    if evt not in G_resolved.nodes():
        G_resolved.add_node(
            evt,
            entity_type="EVENT",
            color=TYPE_COLORS["EVENT"],
        )
        ent_type_map[evt] = "EVENT"
        added += 1
        print(f"  Added to KG: '{evt}'")

print(f"\n✓ Added {added} EVENT nodes to KG")
print(f"KG now: {G_resolved.number_of_nodes()} nodes, "
      f"{G_resolved.number_of_edges()} edges")

# Also connect events to locations via co-occurrence sentences
print("\nAdding event→location edges from co-occurrence...")
edges_added = 0
for evt in event_nodes:
    for sent in doc_resolved.sents:
        if evt in sent.text:
            # Find any GPE/LOC in same sentence
            for ent in sent.ents:
                if ent.label_ in {"GPE", "LOC"} and ent.text.strip() != evt:
                    place = ent.text.strip()
                    if place in G_resolved.nodes():
                        if not G_resolved.has_edge(evt, place):
                            G_resolved.add_edge(
                                evt, place,
                                relations={"OCCURRED_AT"},
                                weight=1,
                            )
                            edges_added += 1
                            print(f"  {evt} → OCCURRED_AT → {place}")

print(f"\n✓ Added {edges_added} event→location edges")

# ── Re-run event linking ──────────────────────────────────────────────────────
print(f"\nEvent nodes to link: {event_nodes}")
print("Linking events to locations...")

for place in geo_df["name"]:
    mentions = [
        s.text.strip()
        for s in doc_resolved.sents
        if place in s.text
    ][:3]

    # Rule-based — check if event name appears in context
    combined = " ".join(mentions).lower()
    rule_matches = [
        evt for evt in event_nodes
        if evt.lower() in combined
    ]
    #
    # # LLM pass on top
    # llm_matches = link_events_to_location_llm(
    #     place, event_nodes, mentions
    # )

    # Merge
    final = list(set(rule_matches))
    geo_df.loc[geo_df["name"] == place, "events"] = str(final)
    print(f"  {place:22s} → {final}")

geo_df["events_list"] = geo_df["events"].apply(
    lambda x: ast.literal_eval(x)
    if isinstance(x, str) and x.startswith("[") else []
)

print(f"\n✓ Event linking complete")
print(geo_df[["name", "events_list"]])

In [ ]:
# ── Leaflet Event-Location Map ────────────────────────────────────────────────
import json, re, ast
from IPython.display import HTML

# ── Build event colour map ────────────────────────────────────────────────────
EVENT_PALETTE = [
    "#E05C5C","#9B59B6","#F0AD4E","#1ABC9C",
    "#E67E22","#3498DB","#E91E63","#00BCD4",
    "#2ECC71","#E74C3C","#8E44AD","#16A085",
]
event_color_map = {
    e: EVENT_PALETTE[i % len(EVENT_PALETTE)]
    for i, e in enumerate(event_nodes)
}

# ── Build event → locations index ─────────────────────────────────────────────
event_location_index = {e: [] for e in event_nodes}

# ── Build locations_js ────────────────────────────────────────────────────────
TYPE_COLOR = {"GPE":"#2563EB","LOC":"#16A34A","FAC":"#D97706"}

locations_js = []
for _, row in geo_df.iterrows():
    etype    = ent_type_map.get(row["name"], "GPE")
    color    = TYPE_COLOR.get(etype, "#6B7280")
    evts     = row.get("events_list", [])
    if isinstance(evts, float):
        evts = []

    # KG relations
    relations = []
    for u, v, d in G_resolved.edges(data=True):
        if u == row["name"] or v == row["name"]:
            rel   = ", ".join(d.get("relations", {"RELATED"}))
            other = v if u == row["name"] else u
            relations.append(f"{other} [{rel}]")

    # Context sentences
    mentions = [
        s.text.strip()
        for s in doc_resolved.sents
        if row["name"] in s.text
    ][:2]

    locations_js.append({
        "name":      row["name"],
        "lat":       float(row["lat"]),
        "lon":       float(row["lon"]),
        "type":      etype,
        "color":     color,
        "summary":   str(row.get("summary","No summary available.")),
        "events":    evts,
        "relations": relations[:5],
        "mentions":  mentions,
        "event_colors": {
            e: event_color_map.get(e,"#888") for e in evts
        },
    })

    # Populate index
    for e in evts:
        if e in event_location_index:
            event_location_index[e].append(row["name"])

# Remove events with no locations
event_location_index = {
    e: locs for e, locs in event_location_index.items() if locs
}
active_events = list(event_location_index.keys())

print(f"✓ {len(locations_js)} locations")
print(f"✓ {len(active_events)} events with linked locations")
for e, locs in event_location_index.items():
    print(f"   {e}: {locs}")

locs_json         = json.dumps(locations_js)
event_nodes_json  = json.dumps(active_events)
event_colors_json = json.dumps(event_color_map)
event_idx_json    = json.dumps(event_location_index)

# ── Render map ────────────────────────────────────────────────────────────────
html = f"""
<html>
<head>
<meta charset='utf-8'/>
<link rel='stylesheet'
  href='https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css'/>
<style>
* {{ box-sizing:border-box; margin:0; padding:0 }}
body {{ font-family:Arial,sans-serif; background:#fff; }}

/* ── Top bar ── */
#topbar {{
  background:#fff; border-bottom:1px solid #e5e7eb;
  padding:8px 14px; display:flex; align-items:center;
  gap:8px; flex-wrap:wrap;
}}
#topbar h2 {{
  font-size:13px; font-weight:700; color:#111827; flex-shrink:0;
}}
#event-filters {{ display:flex; gap:5px; flex-wrap:wrap; flex:1; }}
.evt-btn {{
  font-size:11px; padding:3px 11px; border-radius:12px;
  border:2px solid transparent; cursor:pointer;
  font-weight:600; transition:all .15s; white-space:nowrap;
}}
.evt-btn.active  {{ color:#fff !important; }}
.evt-btn.inactive {{
  background:#f3f4f6 !important;
  color:#9ca3af !important;
  border-color:#e5e7eb !important;
}}
#btn-all {{
  font-size:11px; padding:3px 11px; border-radius:12px;
  border:1px solid #d1d5db; background:#111827;
  color:#fff; cursor:pointer; font-weight:600; white-space:nowrap;
}}

/* ── Layout ── */
#main {{
  display:flex; height:540px; position:relative; overflow:hidden;
}}

/* ── Sidebar ── */
#sidebar {{
  width:220px; flex-shrink:0; background:#fff;
  border-right:1px solid #e5e7eb;
  display:flex; flex-direction:column; overflow:hidden;
}}
#sidebar-head {{
  padding:9px 12px; border-bottom:1px solid #e5e7eb;
  display:flex; align-items:center; justify-content:space-between;
}}
#sidebar-head span {{
  font-size:11px; font-weight:700; text-transform:uppercase;
  letter-spacing:.07em; color:#374151;
}}
#loc-count {{
  font-size:11px; color:#6b7280; font-weight:400;
}}
#loc-search {{
  margin:7px 8px; padding:5px 8px;
  background:#f9fafb; border:1px solid #d1d5db;
  border-radius:5px; font-size:12px;
  color:#111827; outline:none; width:calc(100% - 16px);
}}
#loc-list {{ overflow-y:auto; flex:1; }}
.loc-item {{
  padding:9px 12px; border-bottom:1px solid #f3f4f6;
  cursor:pointer; transition:background .1s;
}}
.loc-item:hover,.loc-item.active {{ background:#f0f4ff; }}
.loc-name {{ font-size:12px; font-weight:700; color:#111827; }}
.loc-type {{
  display:inline-block; font-size:10px; font-weight:600;
  padding:1px 6px; border-radius:8px; margin-top:3px;
}}
.loc-evts {{
  display:flex; flex-wrap:wrap; gap:3px; margin-top:5px;
}}
.evt-tag {{
  font-size:9px; padding:1px 6px; border-radius:6px;
  color:#fff; font-weight:600; white-space:nowrap;
}}

/* ── Map ── */
#map {{ flex:1; }}

/* ── Info panel ── */
#info-panel {{
  position:absolute; bottom:16px; right:12px; z-index:1000;
  background:#fff; border:1px solid #e5e7eb; border-radius:10px;
  padding:14px 16px; width:290px;
  box-shadow:0 6px 24px rgba(0,0,0,.12);
  display:none; max-height:420px; overflow-y:auto;
}}
#info-close {{
  position:absolute; top:8px; right:10px;
  background:none; border:none; font-size:15px;
  color:#9ca3af; cursor:pointer; line-height:1;
}}
.info-name {{
  font-size:14px; font-weight:700; margin-bottom:3px; padding-right:20px;
}}
.info-meta {{
  font-size:10px; color:#6b7280; margin-bottom:8px;
}}
.info-section-title {{
  font-size:9px; font-weight:700; text-transform:uppercase;
  letter-spacing:.08em; color:#9ca3af; margin:9px 0 4px;
}}
.info-summary {{
  font-size:11px; line-height:1.6; color:#374151;
  border-left:3px solid #e5e7eb; padding-left:8px;
  font-style:italic;
}}
.info-evt-list {{
  display:flex; flex-wrap:wrap; gap:4px;
}}
.info-evt-chip {{
  font-size:10px; padding:2px 9px; border-radius:10px;
  color:#fff; font-weight:600; cursor:pointer;
  transition:opacity .15s;
}}
.info-evt-chip:hover {{ opacity:.8; }}
.info-rel-list {{
  font-size:11px; color:#374151;
  padding-left:14px; margin:0; line-height:1.9;
}}
.info-ctx {{
  font-size:11px; line-height:1.5; color:#6b7280;
  border-left:2px solid #f3f4f6; padding-left:7px; margin-top:3px;
}}

/* ── Event legend ── */
#evt-legend {{
  position:absolute; top:10px; right:10px; z-index:1000;
  background:rgba(255,255,255,.95); border:1px solid #e5e7eb;
  border-radius:8px; padding:9px 12px; min-width:175px;
  box-shadow:0 2px 8px rgba(0,0,0,.08);
}}
#evt-legend h4 {{
  font-size:10px; text-transform:uppercase;
  letter-spacing:.08em; color:#6b7280; margin-bottom:7px;
}}
.legend-row {{
  display:flex; align-items:center; gap:7px;
  font-size:11px; color:#374151;
  margin-bottom:5px; cursor:pointer; transition:opacity .15s;
}}
.legend-row:hover {{ opacity:.7; }}
.legend-dot {{
  width:11px; height:11px; border-radius:50%; flex-shrink:0;
}}
.legend-count {{
  font-size:10px; color:#9ca3af; margin-left:auto;
}}

/* ── Arc pulse animation ── */

</style>
</head>
<body>

<!-- Top filter bar -->
<div id='topbar'>
  <h2>🗺 Event–Location Map</h2>
  <button id='btn-all' onclick='showAll()'>Show All</button>
  <div id='event-filters'></div>
</div>

<!-- Map + sidebar -->
<div id='main'>

  <div id='sidebar'>
    <div id='sidebar-head'>
      <span>Locations</span>
      <span id='loc-count'>0</span>
    </div>
    <input id='loc-search' placeholder='Search location or event…'/>
    <div id='loc-list'></div>
  </div>

  <div id='map'></div>

  <!-- Event legend -->
  <div id='evt-legend'>
    <h4>Events</h4>
    <div id='legend-items'></div>
  </div>

  <!-- Info panel -->
  <div id='info-panel'>
    <button id='info-close' onclick='closeInfo()'>✕</button>
    <div id='info-body'></div>
  </div>

</div>

<script src='https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js'>
</script>
<script>
// ── Data ──────────────────────────────────────────────────────────────────────
const LOCS         = {locs_json};
const EVENT_NODES  = {event_nodes_json};
const EVENT_COLORS = {event_colors_json};
const EVENT_IDX    = {event_idx_json};

// ── Map init ──────────────────────────────────────────────────────────────────
const map = L.map('map', {{ zoomControl:true }}).setView([25,15], 2);

const lightTile = L.tileLayer(
  'https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png',
  {{ attribution:'© OpenStreetMap © CARTO', maxZoom:19 }}
).addTo(map);

const streetTile = L.tileLayer(
  'https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',
  {{ attribution:'© OpenStreetMap contributors', maxZoom:19 }}
);

L.control.layers(
  {{ 'Light (CartoDB)':lightTile, 'Street (OSM)':streetTile }},
  null, {{ position:'bottomleft' }}
).addTo(map);

// ── State ─────────────────────────────────────────────────────────────────────
let activeEvent  = null;
let arcLayers    = [];
let currentLocs  = [...LOCS];
const markers    = {{}};

// ── Build markers ─────────────────────────────────────────────────────────────
LOCS.forEach(loc => {{
  const hasEvts = loc.events && loc.events.length > 0;

  const icon = L.divIcon({{
    className: '',
    html: `<div style="
      width:15px; height:15px; border-radius:50%;
      background:${{loc.color}}; border:2.5px solid #fff;
      box-shadow:0 1px 6px rgba(0,0,0,.3);
      ${{hasEvts
        ? 'outline:3px solid '+loc.color+';outline-offset:2px;'
        : ''}}
    "></div>`,
    iconSize:[15,15], iconAnchor:[7,7], popupAnchor:[0,-10]
  }});

  const m = L.marker([loc.lat, loc.lon], {{ icon }})
    .bindTooltip(
      `<b>${{loc.name}}</b>
       ${{hasEvts
         ? '<br><span style="font-size:10px;color:#555">'
           + (loc.events||[]).join(', ') + '</span>'
         : ''}}`,
      {{ direction:'top', offset:[0,-10] }}
    )
    .on('click', () => {{
      showInfo(loc);
      // Highlight this marker
      document.querySelectorAll('.loc-item')
        .forEach(e => e.classList.remove('active'));
      const el = document.querySelector(
        `.loc-item[data-name="${{loc.name}}"]`
      );
      if (el) el.classList.add('active');
    }});

  m.addTo(map);
  markers[loc.name] = m;
}});

// ── Arc helpers ───────────────────────────────────────────────────────────────
function clearArcs() {{
  arcLayers.forEach(a => map.removeLayer(a));
  arcLayers = [];
}}

function drawArc(loc1, loc2, color) {{
  // Slight midpoint elevation for arc curve effect
  const midLat = (loc1.lat + loc2.lat) / 2 + 8;
  const midLon = (loc1.lon + loc2.lon) / 2;
  const arc = L.polyline(
    [[loc1.lat,loc1.lon],[midLat,midLon],[loc2.lat,loc2.lon]],
    {{
      color,
      weight:    2.5,
      opacity:   0.8,
      dashArray: '7,5',
      className: 'arc-pulse',
    }}
  ).addTo(map);
  arcLayers.push(arc);
}}

// ── Highlight event ───────────────────────────────────────────────────────────
function highlightEvent(eventName) {{
  clearArcs();
  closeInfo();
  activeEvent     = eventName;
  const color     = EVENT_COLORS[eventName] || '#888';
  const evtLocs   = EVENT_IDX[eventName]    || [];

  // Dim non-event markers
  Object.entries(markers).forEach(([name, m]) =>
    m.setOpacity(evtLocs.includes(name) ? 1.0 : 0.1)
  );

  // Draw arcs between all locations of this event
  const coords = evtLocs
    .map(n => LOCS.find(l => l.name === n))
    .filter(Boolean);
  for (let i=0; i<coords.length; i++)
    for (let j=i+1; j<coords.length; j++)
      drawArc(coords[i], coords[j], color);

  // Fit map to event locations
  if (coords.length > 1) {{
    const bounds = L.latLngBounds(coords.map(c => [c.lat,c.lon]));
    map.fitBounds(bounds, {{ padding:[40,40] }});
  }} else if (coords.length === 1) {{
    map.flyTo([coords[0].lat, coords[0].lon], 5);
  }}

  // Update sidebar
  currentLocs = LOCS.filter(l => evtLocs.includes(l.name));
  renderList(currentLocs);

  // Update filter buttons
  document.querySelectorAll('.evt-btn').forEach(b => {{
    const on = b.dataset.event === eventName;
    b.classList.toggle('active',   on);
    b.classList.toggle('inactive', !on);
    if (on) {{
      b.style.background   = color;
      b.style.borderColor  = color;
    }} else {{
      b.style.background   = '';
      b.style.borderColor  = '';
    }}
  }});

  // Update legend
  document.querySelectorAll('.legend-row').forEach(r => {{
    r.style.opacity = r.dataset.event === eventName ? '1' : '0.35';
  }});
}}

// ── Show all ──────────────────────────────────────────────────────────────────
function showAll() {{
  clearArcs();
  closeInfo();
  activeEvent = null;
  currentLocs = [...LOCS];

  Object.values(markers).forEach(m => m.setOpacity(1));
  map.setView([25,15], 2);

  document.querySelectorAll('.evt-btn').forEach(b => {{
    b.classList.remove('active','inactive');
    b.style.background  = '';
    b.style.borderColor = '';
  }});
  document.querySelectorAll('.legend-row')
    .forEach(r => r.style.opacity = '1');

  renderList(currentLocs);
}}

// ── Info panel ────────────────────────────────────────────────────────────────
function showInfo(loc) {{
  const panel = document.getElementById('info-panel');
  const color = loc.color;

  // Events section
  const evtsHtml = loc.events && loc.events.length
    ? `<div class='info-section-title'>Linked Events</div>
       <div class='info-evt-list'>
         ${{(loc.events||[]).map(e =>
           `<span class='info-evt-chip'
             style='background:${{EVENT_COLORS[e]||"#888"}}'
             onclick='highlightEvent("${{e}}")'
             title='Click to highlight event'>${{e}}</span>`
         ).join('')}}
       </div>`
    : `<div class='info-section-title'>Linked Events</div>
       <div style='font-size:11px;color:#9ca3af'>
         No events linked to this location
       </div>`;

  // Relations section
  const relsHtml = loc.relations && loc.relations.length
    ? `<div class='info-section-title'>KG Relations</div>
       <ul class='info-rel-list'>
         ${{(loc.relations||[]).map(r=>`<li>${{r}}</li>`).join('')}}
       </ul>`
    : '';

  // Context sentences
  const ctxHtml = loc.mentions && loc.mentions.length
    ? `<div class='info-section-title'>Source Text</div>
       ${{loc.mentions.map(m=>
         `<div class='info-ctx'>${{m}}</div>`
       ).join('')}}`
    : '';

  document.getElementById('info-body').innerHTML = `
    <div class='info-name' style='color:${{color}}'>${{loc.name}}</div>
    <div class='info-meta'>
      ${{loc.type}} &nbsp;·&nbsp;
      ${{loc.lat.toFixed(4)}}°, ${{loc.lon.toFixed(4)}}°
    </div>
    <div class='info-section-title'>Summary</div>
    <div class='info-summary'>${{loc.summary}}</div>
    ${{evtsHtml}}
    ${{relsHtml}}
    ${{ctxHtml}}`;

  panel.style.display = 'block';
}}

function closeInfo() {{
  document.getElementById('info-panel').style.display = 'none';
}}

// ── Sidebar list ──────────────────────────────────────────────────────────────
const listEl   = document.getElementById('loc-list');
const countEl  = document.getElementById('loc-count');
const searchEl = document.getElementById('loc-search');

function renderList(locs) {{
  listEl.innerHTML = '';
  countEl.textContent = locs.length;

  locs.forEach(loc => {{
    const el = document.createElement('div');
    el.className   = 'loc-item';
    el.dataset.name = loc.name;

    const evtTags = (loc.events||[]).map(e =>
      `<span class='evt-tag'
        style='background:${{EVENT_COLORS[e]||"#888"}}'>${{
          e.length > 18 ? e.slice(0,16)+'…' : e
        }}</span>`
    ).join('');

    el.innerHTML = `
      <div class='loc-name'>${{loc.name}}</div>
      <span class='loc-type'
        style='background:${{loc.color}}18;color:${{loc.color}}'>
        ${{loc.type}}
      </span>
      ${{evtTags
        ? `<div class='loc-evts'>${{evtTags}}</div>`
        : `<div style='font-size:10px;color:#d1d5db;margin-top:4px'>
             no events linked</div>`
      }}`;

    el.onclick = () => {{
      document.querySelectorAll('.loc-item')
        .forEach(e => e.classList.remove('active'));
      el.classList.add('active');
      map.flyTo([loc.lat, loc.lon], 6, {{ animate:true, duration:1.2 }});
      setTimeout(() => showInfo(loc), 1000);
    }};

    listEl.appendChild(el);
  }});
}}

// ── Search ────────────────────────────────────────────────────────────────────
searchEl.addEventListener('input', e => {{
  const q    = e.target.value.toLowerCase();
  const base = activeEvent
    ? LOCS.filter(l => (EVENT_IDX[activeEvent]||[]).includes(l.name))
    : [...LOCS];
  renderList(q
    ? base.filter(l =>
        l.name.toLowerCase().includes(q) ||
        (l.events||[]).some(ev => ev.toLowerCase().includes(q)) ||
        (l.summary||'').toLowerCase().includes(q)
      )
    : base
  );
}});

// ── Event filter buttons ──────────────────────────────────────────────────────
const filterDiv = document.getElementById('event-filters');
EVENT_NODES.forEach(evt => {{
  const color = EVENT_COLORS[evt] || '#888';
  const locs  = EVENT_IDX[evt]   || [];
  const btn   = document.createElement('button');
  btn.className     = 'evt-btn';
  btn.dataset.event = evt;
  btn.textContent   = `${{evt}} (${{locs.length}})`;
  btn.style.background  = color + '20';
  btn.style.borderColor = color;
  btn.style.color       = color;
  btn.onclick = () =>
    activeEvent === evt ? showAll() : highlightEvent(evt);
  filterDiv.appendChild(btn);
}});

// ── Legend ────────────────────────────────────────────────────────────────────
const legendDiv = document.getElementById('legend-items');
EVENT_NODES.forEach(evt => {{
  const color = EVENT_COLORS[evt] || '#888';
  const locs  = EVENT_IDX[evt]   || [];
  const row   = document.createElement('div');
  row.className    = 'legend-row';
  row.dataset.event = evt;
  row.innerHTML = `
    <div class='legend-dot' style='background:${{color}}'></div>
    <span style='flex:1'>${{evt.length>22?evt.slice(0,20)+'…':evt}}</span>
    <span class='legend-count'>${{locs.length}}</span>`;
  row.onclick = () =>
    activeEvent === evt ? showAll() : highlightEvent(evt);
  legendDiv.appendChild(row);
}});

// ── Initial render ────────────────────────────────────────────────────────────
renderList(LOCS);
</script>
</body></html>
"""

HTML(html)